In [37]:
import pandas as pd
import json

In [38]:
original_path = "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info.json"
scale_folder = "../outputs/scale-exp"

In [39]:
import glob
glob.glob(scale_folder + "/base/*/*")[:5]

['../outputs/scale-exp/base/prompt_0/token_113',
 '../outputs/scale-exp/base/prompt_0/token_114',
 '../outputs/scale-exp/base/prompt_0/token_115',
 '../outputs/scale-exp/base/prompt_0/token_116',
 '../outputs/scale-exp/base/prompt_0/token_117']

In [40]:
base_folder = scale_folder + "/base/"
instruct_folder = scale_folder + "/instruct/"

In [43]:
from collections import Counter

def any_repeated_substring(s, min_repeats=5, max_sub_len=10):
    n = len(s)
    L = max_sub_len
    counts = Counter(s[i:i+L] for i in range(n - L + 1))
    if any(v >= min_repeats for v in counts.values()):
        # print("The max repeating subsequence is - ",counts.most_common(1)[0][0])
        return True
    return False


def _downgrade_plan_to_degenerate(steered_text, original_ym = None):
    """
        - empty string, single character, single string, then classify sa degen.
        - multiple number of 'import' statements in the output without any return statement at the end.
    """
    ## if original_ym is part of the steered text, it's marked as degenerate to prevent a False Positive.
    if original_ym != None and original_ym in steered_text:
        return False

    stripped_text = steered_text.strip()
    if stripped_text == "" or " " not in stripped_text:
        return True
    if any_repeated_substring(steered_text) and "return" not in steered_text:
        return True
    
    return False

In [44]:
def _extract_code_part(input_prefix_text, suffix_text):
    full_text = input_prefix_text + suffix_text
    return "def" + full_text.split("```python\n")[1].split("def")[1]

def _classify_as_planning_vs_not_planning(metadata_json, steering_results, planning_analysis):
    """
        Logic:
            - if already marked as 'Not planning', we keep as-is.
            - if marked as planning, we check for non-degeneracy.
            - if marked as can't say, we check for non-degenaracy and whether the original and new normalized sequences differ by much.
        
        Returns, new planning analysis, with 
        {
            "y_m": {
                "original_verdict": 
                "new_verdict":
                "base_similarity_score":
            }
            ....
        }
    """
    ym_keys = list(planning_analysis.keys())
    new_planning_analysis = {}
    for y_m in ym_keys:
        new_verdict = planning_analysis[y_m]
        input_prefix_part = metadata_json["input_prefix_text"]
        base_suffix = steering_results[y_m]["base_text"]
        base_code = _extract_code_part(input_prefix_part, base_suffix)
        
        if y_m == "return":
            new_verdict = "Not planning"
        elif planning_analysis[y_m] == "Plan":
            ## not base_suffix.startswith(y_m) to take care of the y_m which are just after in the generation -> a bug in the existing pipeline.            
            if not base_suffix.startswith(y_m) and not all(e["steered_text"] == "" or _downgrade_plan_to_degenerate(e["decoded_text"], y_m) for e in steering_results[y_m]["steered"]):
                new_verdict = "Plan"
            else:
                new_verdict = "Can't say"                
        
        new_planning_analysis[y_m] = {
            "original_verdict": planning_analysis[y_m],
            "new_verdict": new_verdict
        }
    
    return new_planning_analysis

In [45]:
all_tokens = [*glob.glob(scale_folder + "/base/*/*"), *glob.glob(scale_folder + "/instruct/*/*")]

In [46]:
def load_json(file):
    with open(file, "r") as f:
        return json.load(f)

In [47]:
glob.glob(f"{all_tokens[0]}/*.json")

['../outputs/scale-exp/base/prompt_0/token_113/clusters.json',
 '../outputs/scale-exp/base/prompt_0/token_113/metadata.json',
 '../outputs/scale-exp/base/prompt_0/token_113/planning_analysis.json',
 '../outputs/scale-exp/base/prompt_0/token_113/steering_results.json']

In [48]:
def dump_json(filename, my_dict):
    with open(filename, "w") as f:
        json.dump(my_dict, f, indent=4)  # indent=4 makes it pretty-printed

In [49]:
import os

for token_planning_folder in all_tokens:
    json_files = glob.glob(f"{token_planning_folder}/*.json")
    metadata, planning_analysis, steering_results = load_json(json_files[1]), load_json(json_files[2]), load_json(json_files[3])
    new_planning_analysis = _classify_as_planning_vs_not_planning(metadata, steering_results, planning_analysis)
    updated_folder = token_planning_folder.replace("scale-exp", "t-scale-exp-v2")
    os.makedirs(updated_folder, exist_ok=True)
    dump_json(updated_folder + "/updated_planning_analysis.json", new_planning_analysis)

In [50]:
og_pass_file = "../data/external/first_100_passing_examples_without_docstrings_base_model_og_prompt_V2.json"
og_fail_file = "../data/external/first_100_failing_examples_without_docstrings_base_model_og_prompt_V2.json"
og_pass = load_json(og_pass_file)
og_fail = load_json(og_fail_file)
og_pass_ids = [entry["task_id"] for entry in og_pass]
og_fail_ids = [entry["task_id"] for entry in og_fail]

In [51]:
selected_file = original_path
selected_data = load_json(selected_file)
selected_ids = [entry["task_id"] for entry in selected_data]

In [52]:
### setting whether base has passed OR failed here.
for entry in selected_data:
    if entry["task_id"] in og_pass_ids:
        entry["base_pass"] = True
    else:
        entry["base_pass"] = False

In [53]:
glob.glob("../outputs/t-scale-exp-v2/base/prompt_0/*/*.json")

['../outputs/t-scale-exp-v2/base/prompt_0/token_113/updated_planning_analysis.json',
 '../outputs/t-scale-exp-v2/base/prompt_0/token_114/updated_planning_analysis.json',
 '../outputs/t-scale-exp-v2/base/prompt_0/token_115/updated_planning_analysis.json',
 '../outputs/t-scale-exp-v2/base/prompt_0/token_116/updated_planning_analysis.json',
 '../outputs/t-scale-exp-v2/base/prompt_0/token_117/updated_planning_analysis.json']

In [54]:
def _detect_planning(planning_dict):
    keys = []
    for key, value in planning_dict.items():
        if value["new_verdict"] == "Plan":
            keys.append(key)
    return keys

def _detect_cant_says(planning_dict):
    present = False
    for key, value in planning_dict.items():
        if value["new_verdict"] == "Can't say":
            present = True
    return present

def _get_ym_plans(folder):
    token_planning_files = glob.glob(folder + "/*/*.json")
    planning_datas = [load_json(f) for f in token_planning_files]
    y_ms = []
    for data in planning_datas:
        y_ms.extend(_detect_planning(data))
    y_ms = list(set(y_ms))
    return y_ms

def _detect_cs(folder):
    token_planning_files = glob.glob(folder + "/*/*.json")
    planning_datas = [load_json(f) for f in token_planning_files]
    for data in planning_datas:
        if _detect_cant_says(data):
            return True
    return False

def _get_base_and_instruct_plans(iter):
    base_folder = f"../outputs/t-scale-exp-v2/base/prompt_{iter}"
    instruct_folder = f"../outputs/t-scale-exp-v2/instruct/prompt_{iter}"
    if os.path.exists(base_folder) and os.path.exists(instruct_folder):
        base_yms = _get_ym_plans(base_folder)
        instruct_yms = _get_ym_plans(instruct_folder)
    else:
        base_yms = None
        instruct_yms = None
    return {
        "base": base_yms,
        "instruct": instruct_yms
    }

def _get_base_and_instruct_cantsays(iter):
    base_folder = f"../outputs/t-scale-exp-v2/base/prompt_{iter}"
    instruct_folder = f"../outputs/t-scale-exp-v2/instruct/prompt_{iter}"
    if os.path.exists(base_folder):
        base_cs = _detect_cs(base_folder)
    else:
        base_cs = False
    if os.path.exists(instruct_folder):
        instruct_cs = _detect_cs(instruct_folder)
    else:
        instruct_cs = False
    return {
        "base": base_cs,
        "instruct": instruct_cs
    }

In [55]:
all_prompts = range(len(selected_ids))
len(all_prompts)

81

In [56]:
for iter, entry in enumerate(selected_data):
    ym_plans = _get_base_and_instruct_plans(iter)
    entry["base_plans"] = ym_plans["base"]
    entry["instruct_plans"] = ym_plans["instruct"]
    cs = _get_base_and_instruct_cantsays(iter)    
    entry["base_cs"] = cs["base"]
    entry["instruct_cs"] = cs["instruct"]

In [57]:
# --- TABLE 1: ALL CASES ---
# Rows: instruct plans / does not plan
# Columns: base pass / base fail
table1 = {
    True:  {True: 0, False: 0},  # instruct plans
    False: {True: 0, False: 0}   # instruct does not plan
}

# --- TABLE 2: ONLY WHEN INSTRUCT PLANS ---
# Rows: base plans / base does not plan
# Columns: base pass / base fail
table2 = {
    True:  {True: 0, False: 0},  # base plans
    False: {True: 0, False: 0}   # base does not plan
}

# --- TABLE 3: ONLY WHEN INSTRUCT DOES *NOT* PLAN ---
# Rows: base plans / base does not plan
# Columns: base pass / base fail
table3 = {
    True:  {True: 0, False: 0},  # base plans
    False: {True: 0, False: 0}   # base does not plan
}

# --- TABLE 4: CONFUSION MATRIX OF INSTRUCT PLANS vs BASE PLANS ---
# Rows: instruct plans / instruct does not plan
# Columns: base plans / base does not plan
table4 = {
    True:  {True: 0, False: 0},  # instruct plans
    False: {True: 0, False: 0}   # instruct does not plan
}

for entry in selected_data:

    base_pass = bool(entry.get("base_pass"))
    base_plans_empty = isinstance(entry.get("base_plans"), list) and len(entry["base_plans"]) == 0
    instruct_plans_empty = isinstance(entry.get("instruct_plans"), list) and len(entry["instruct_plans"]) == 0

    ## skip cases which are pure "Can't Says" (No plan but containing Can't Says)
    if base_plans_empty and entry["base_cs"] == True:
        print("skipping entry because of base")
        continue

    if instruct_plans_empty and entry["instruct_cs"] == True:
        print("skipping entry because of instruct")
        continue

    instruct_plans = not instruct_plans_empty
    base_plans = not base_plans_empty

    # Table 1
    table1[instruct_plans][base_pass] += 1

    # Table 2
    if instruct_plans:
        table2[base_plans][base_pass] += 1

    # Table 3
    else:
        table3[base_plans][base_pass] += 1

    # --- Table 4: confusion matrix (instruct plans vs base plans) ---
    table4[instruct_plans][base_plans] += 1


# Pretty print

print("\n=== TABLE 1: ALL CASES ===")
print("                    Base Pass | Base Fail")
print("------------------------------------------")
print(f"Instruct Plans       {table1[True][True]:9d} | {table1[True][False]:9d}")
print(f"Instruct No-Plan     {table1[False][True]:9d} | {table1[False][False]:9d}")

print("\n=== TABLE 2: ONLY CASES WHERE INSTRUCT PLANS ===")
print("                     Base Pass | Base Fail")
print("-------------------------------------------")
print(f"Base Plans           {table2[True][True]:9d} | {table2[True][False]:9d}")
print(f"Base No-Plan         {table2[False][True]:9d} | {table2[False][False]:9d}")

print("\n=== TABLE 3: ONLY CASES WHERE INSTRUCT DOES NOT PLAN ===")
print("                     Base Pass | Base Fail")
print("-------------------------------------------")
print(f"Base Plans           {table3[True][True]:9d} | {table3[True][False]:9d}")
print(f"Base No-Plan         {table3[False][True]:9d} | {table3[False][False]:9d}")

print("\n=== TABLE 4: CONFUSION MATRIX (INSTRUCT PLANS vs BASE PLANS) ===")
print("                     Base Plans | Base No-Plan")
print("------------------------------------------------")
print(f"Instruct Plans       {table4[True][True]:11d} | {table4[True][False]:11d}")
print(f"Instruct No-Plan     {table4[False][True]:11d} | {table4[False][False]:11d}")

skipping entry because of base
skipping entry because of instruct
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of instruct
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of instruct
skipping entry because of instruct
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of instruct
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of base
skipping entry because of instruct
skipping entry because of base

=== TABLE 1: ALL CASES ===
                    Base Pass | Base Fail
------------------------------------------
Instruct Plans              20 |        15
Instruct No-Plan            17 |         6

=== TABLE 2: ONLY CASES WHERE INSTRUCT PLANS ===
              

In [58]:
import json

with open(
    "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json",
    "w"
) as f:
    json.dump(selected_data, f, indent=2)